In [ ]:
import cv2
import numpy as np
import mediapipe as mp
import pygame

mp_face_mesh = mp.solutions.face_mesh
face_mesh = mp_face_mesh.FaceMesh(static_image_mode=False, max_num_faces=1, min_detection_confidence=0.5)

# Initializing the camera and taking the instance
cap = cv2.VideoCapture(0)

# Status marking for current state
sleep = 0
drowsy = 0
active = 0
status = ""
color = (0, 0, 0)

def compute(ptA, ptB):
    dist = np.linalg.norm(np.array(ptA) - np.array(ptB))
    return dist

def blinked(a, b, c, d, e, f):
    up = compute(b, d) + compute(c, e)
    down = compute(a, f)
    ratio = up / (2.0 * down)

    # Checking the eyes blink frequency/duration
    if ratio > 0.40:
        return 2
    elif ratio > 0.25 and ratio <= 0.40:
        return 1
    else:
        return 0

def yawning(a, b, c, d, e, f, g, h):
    up = compute(b, e) + compute(c, f) + compute(d, g)
    down = compute(a, h)
    ratio_yawn = up / (2.0 * down)

    # Checking if the driver yawned
    if ratio_yawn > 0.30:
        return 1
    else:
        return 0

def play_alarm_sound():
    pygame.mixer.music.load("Alarm.mp3")
    pygame.mixer.music.play()

pygame.init()
pygame.mixer.init()

while True:
    _, frame = cap.read()
    frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    results = face_mesh.process(frame_rgb)

    if results.multi_face_landmarks:
        for face_landmarks in results.multi_face_landmarks:
            landmarks = []
            for landmark in face_landmarks.landmark:
                x, y = int(landmark.x * frame.shape[1]), int(landmark.y * frame.shape[0])
                landmarks.append((x, y))

            left_blink = blinked(landmarks[33], landmarks[160], landmarks[158], landmarks[144], landmarks[153], landmarks[145])
            right_blink = blinked(landmarks[362], landmarks[385], landmarks[387], landmarks[380], landmarks[374], landmarks[386])
            lips_yawn = yawning(landmarks[61], landmarks[39], landmarks[37], landmarks[0], landmarks[267], landmarks[269], landmarks[291], landmarks[375])
            
            if left_blink == 0 or right_blink == 0:
                sleep += 1
                drowsy = 0
                active = 0
                if sleep > 5:
                    status = "SLEEPING !!!"
                    color = (255, 0, 0)
                    play_alarm_sound()
            
            elif left_blink == 1 or right_blink == 1:
                sleep = 0
                active = 0
                drowsy += 1
                if drowsy > 12:
                    status = "Drowsy !"
                    color = (0, 0, 255)
                    play_alarm_sound()

            else:
                drowsy = 0
                sleep = 0
                active += 1
                if active > 5:
                    status = "Active :)"
                    color = (0, 255, 0)

            cv2.putText(frame, status, (100, 100), cv2.FONT_HERSHEY_SIMPLEX, 1.2, color, 3)

            for landmark in landmarks:
                cv2.circle(frame, landmark, 1, (255, 255, 255), -1)

    cv2.imshow("Frame", frame)
    key = cv2.waitKey(1)
    if key == 27:
        break

cv2.destroyAllWindows()
cap.release()
pygame.mixer.music.stop()
pygame.quit()